<a href="https://colab.research.google.com/github/srinivasa04/UKVisaAIAssistant/blob/main/UK_Visa_Guidance_Agent_Capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 2 - Basic Working Agent

Project: UK Visa Application Guidance & Document Readiness Agent

This phase implements a rule-based baseline agent that:
- Accepts user input
- Generates template-based responses
- Logs conversations
- Demonstrates baseline limitations

In [1]:
import os

os.makedirs("logs", exist_ok=True)

print("Project folders created successfully.")

Project folders created successfully.


In [2]:
def get_response(user_input):
    """
    Simple rule-based response generator/agent.
    """

    question = user_input.lower()

    if "student" in question:
        return (
            "A Student visa allows eligible international students "
            "to study in the UK."
        )

    elif "skilled worker" in question:
        return (
            "A Skilled Worker visa generally requires "
            "a job offer from an approved UK sponsor."
        )

    elif "visitor" in question:
        return (
            "A Standard Visitor visa is intended for tourism, "
            "business meetings and short visits."
        )

    elif "documents" in question:
        return (
            "Typical supporting documents include a passport, "
            "application form and other evidence depending on the visa."
        )

    else:
        return (
            "Sorry, I don't understand your question. "
            "This baseline agent only supports predefined topics."
        )

In [7]:
# Create log files to store conversation history.

from datetime import datetime

LOG_FILE = "logs/conversation_log.txt"

def log_conversation(user, response):

    with open(LOG_FILE, "a", encoding="utf-8") as file:

        file.write("=" * 50 + "\n")
        file.write(f"Time: {datetime.now()}\n")
        file.write(f"User : {user}\n")
        file.write(f"Agent: {response}\n\n")

In [10]:
# Building the CLI

print("=" * 60)
print("UK Visa Application Guidance Agent (Baseline)")
print("Type 'exit' to stop.")
print("=" * 60)

while True:

    user_input = input("\nYou: ")

    if user_input.lower() == "exit":
        print("Session ended.")
        break

    response = get_response(user_input)

    print("\nAgent:", response)

    log_conversation(user_input, response)

UK Visa Application Guidance Agent (Baseline)
Type 'exit' to stop.

You: I have received my COS letter.

Agent: Sorry, I don't understand your question. This baseline agent only supports predefined topics.

You: I'm from India and have a UK job offer.

Agent: Sorry, I don't understand your question. This baseline agent only supports predefined topics.

You: Student visa

Agent: A Student visa allows eligible international students to study in the UK.

You: What documents do I need?

Agent: Typical supporting documents include a passport, application form and other evidence depending on the visa.

You: exit
Session ended.


# Phase 3 Architecture

User
   │
Prompt Builder
   │
Gemini API
   │
LLM Response

In [11]:
!pip install -q google-genai

In [ ]:
# Configure the API Key

from google.colab import userdata
from google import genai

client = genai.Client(
    api_key=userdata.get("GOOGLE_API_KEY")
)

In [17]:
response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="Hello"
)

print(response.text)

Hello! How can I help you today?


In [27]:
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GOOGLE_API_KEY"))

MODEL_NAME = "gemini-3.6-flash"

import time
from google.genai.errors import ClientError

def ask_gemini(system_prompt, user_prompt):

    while True:
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=f"""
SYSTEM:
{system_prompt}

USER:
{user_prompt}
"""
            )
            return response.text

        except ClientError as e:
            if "RESOURCE_EXHAUSTED" in str(e):
                print("Rate limit reached. Waiting 30 seconds...")
                time.sleep(30)
            else:
                raise

In [19]:
SYSTEM_PROMPT = """
You are a helpful UK Visa Guidance Assistant.
"""

question = "What is a Skilled Worker visa?"

answer = ask_gemini(SYSTEM_PROMPT, question)

print(answer)

The **Skilled Worker visa** is the primary UK immigration route for non-UK nationals who wish to come to or stay in the UK to work in an eligible skilled job with an approved employer. 

It replaced the older Tier 2 (General) work visa.

---

### **Key Requirements**

To qualify for a Skilled Worker visa, you must meet specific criteria set by the UK Home Office:

1. **Approved Employer:** You must have a job offer from a UK employer that has been approved by the Home Office as a licensed sponsor.
2. **Certificate of Sponsorship (CoS):** Your employer must issue you a valid CoS, which details the role you have been offered.
3. **Eligible Job:** The job must be at an eligible skill level (generally RQF Level 3 or above, equivalent to A-levels).
4. **Minimum Salary:** You must be paid a minimum salary threshold. Following recent updates, the standard general salary threshold is **£38,700 per year** (or the "going rate" for your specific occupation, whichever is higher). 
   * *Note:* Low

In [21]:
# Baseline Prompt V1
PROMPT_V1 = """
You are a UK Visa Guidance Assistant.

Answer the user's questions politely.
"""

In [22]:
# Safer Prompt V2
PROMPT_V2 = """
You are a UK Visa Guidance Assistant.

Rules:
- Answer only UK visa guidance questions.
- Do not provide legal advice.
- If unsure, clearly say so.
- Suggest checking GOV.UK for confirmation.
- Keep answers concise.
"""

In [23]:
# Prompt Version 3 (Safer & detailed)
PROMPT_V3 = """
You are an AI assistant that helps immigration consultants and prospective UK visa applicants understand official UK visa guidance.

Responsibilities:
- Explain UK visa guidance in simple language.
- Ask follow-up questions if important information is missing.
- Never provide legal advice.
- Never guarantee visa approval.
- Never invent visa requirements.
- If uncertain, explain the uncertainty.
- Use bullet points where appropriate.
- Recommend checking official GOV.UK guidance for important decisions.

Tone:
Professional, clear and concise.
"""

In [25]:
test_questions = [

    "I have a UK job offer. Which visa should I consider?",

    "Can my spouse travel with me on a Student visa?",

    "What documents are needed for a Skilled Worker visa?",

    "Will my visa definitely be approved?",

    "I previously overstayed in another country. Should I mention this?"
]

In [ ]:
prompts = {
    "Prompt V1": PROMPT_V1,
    "Prompt V2": PROMPT_V2,
    "Prompt V3": PROMPT_V3,
}

for prompt_name, prompt in prompts.items():

    print("=" * 80)
    print(prompt_name)
    print("=" * 80)

    for question in test_questions:

        print(f"\nUser: {question}\n")

        answer = ask_gemini(prompt, question)

        print(answer)
        time.sleep(15)
        print("-" * 80)

Prompt V1

User: I have a UK job offer. Which visa should I consider?

Congratulations on receiving a job offer in the UK! 

The most common and primary route for working in the UK with a job offer is the **Skilled Worker visa**. However, the exact visa you should apply for depends on the nature of your job, your industry, and your employer. 

Here are the main options you should consider:

---

### 1. Skilled Worker Visa *(Most Common)*
This is the standard visa for foreign nationals coming to work in the UK.
* **Key Requirements:**
  * Your employer must be approved by the UK Home Office as a **licensed sponsor**.
  * They must issue you a **Certificate of Sponsorship (CoS)**.
  * The job must be at an eligible skill level (RQF Level 3 or higher, roughly equivalent to A-levels).
  * You must meet the salary threshold (generally at least £38,700 per year, or the official "going rate" for your job, whichever is higher—though lower thresholds apply to certain roles, PhD holders, or "new